# Figure fix: rebuild Figures 4 and 7 from the evaluation outputs
Run all (CPU is fine). Upload `figure_fix_upload.zip` from your Drive when it finishes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
MY='/content/drive/MyDrive'; RUNS=f'{MY}/pix2pix_runs'; EVAL=f'{RUNS}/evaluation'
OUT=f'{RUNS}/figure_fix'; os.makedirs(OUT, exist_ok=True)
if not os.path.exists('/content/dataset/manifest.csv'):
    zipfile.ZipFile(f'{MY}/dataset.zip').extractall('/content/dataset')
ids=[299,300,301,307,334,335]
def fig(folder, name):
    f,ax=plt.subplots(2,6,figsize=(18,6.2))
    for j,i in enumerate(ids):
        ax[0,j].imshow(Image.open(f'/content/dataset/test/input/tile_{i}.png')); ax[0,j].set_title(f'tile {i}',fontsize=9)
        ax[1,j].imshow(Image.open(f'{EVAL}/{folder}/{i}.png'))
    for a in ax.flat: a.set_axis_off()
    plt.tight_layout(); plt.savefig(f'{OUT}/{name}',dpi=150); plt.show()
fig('fid_cond','fig4_conditional.png'); fig('fid_base_zero','fig7_unconditional.png')

# which source did the published figures come from? compare sample grids with the evaluation outputs
def arr(p, size=(128,128)): return np.asarray(Image.open(p).convert('RGB').resize(size), float)
lines=[]
for i in ids:
    d=np.abs(arr(f'{EVAL}/fid_cond/{i}.png')-arr(f'{EVAL}/fid_base_zero/{i}.png')).mean()
    lines.append(f'tile {i}: mean abs pixel difference conditional vs unconditional = {d:.1f}')
for f in ['conditional_samples_final.png','baseline_samples_final.png','baseline_collapsed_samples_final.png']:
    p=f'{RUNS}/{f}'
    if os.path.exists(p): shutil.copy(p, f'{OUT}/{f}')
    lines.append(f'{f}: {"copied" if os.path.exists(p) else "missing"}')
open(f'{OUT}/figure_fix_check.txt','w').write('\n'.join(lines)); print('\n'.join(lines))
print('saved', shutil.make_archive(f'{MY}/figure_fix_upload','zip',OUT))